https://github.com/YuqianTrudyLi/Projects/blob/main/Down-and-out%20Barrier%20Call%20Option%20Pricing.ipynb

# <span style="color:blue">Method 3: Closed-form analytical method of PDE </span>

$$
v(t,x) = c_K(t,x) - (\frac{x}{B})^{2\alpha}c_K(t,\frac{B^2}{x})
$$

where $c_K(t,x)$ is the BS European call of strike $K$ and maturity $T$ if $S_t = x$, interest rate is $r$, and $\alpha = \frac{1}{2}(1-\frac{2r}{\sigma^2})$ 

In [67]:
import numpy as np

In [68]:
def EurCall(t,T,r,sigma,x,K):
    from scipy.stats import norm
    d1 = (np.log(x/K)+(r+sigma**2/2)*(T-t))/(sigma*np.sqrt(T-t))
    d2 = (np.log(x/K)+(r-sigma**2/2)*(T-t))/(sigma*np.sqrt(T-t))
    premium = x*norm.cdf(d1)-K*np.exp(-r*(T-t))*norm.cdf(d2)
    return premium

In [69]:
def Analytical(t,T,r,sigma,x,B,K):
    return EurCall(t,T,r,sigma,x,K)-(x/(B+1e-6))**(1-2*r/(sigma**2))*EurCall(t,T,r,sigma,B**2/x,K)

In [70]:
# example BS formula
EurCall(0,1,0.01,0.2,100,100)

np.float64(8.433318690109608)

In [71]:
Analytical(0,1,0.01,0.2,100,90,100)      #barrier at 0 should give same result as BS
#increasing the volatility, the price of the barrier decreases (wrt the vanilla)

np.float64(6.876126707866938)

In [72]:
#convergence to L->0 or L->S0
for l in np.range(0,100):
    #k=110
    EurCall(0,1,0.01,0.2,l,110)

AttributeError: module 'numpy' has no attribute 'range'

- Change set of parameters: x=S_0, K, B, T, r, sigma

- Verify that the price is lower than the vanilla call -> OK

In [ ]:
# down and in http://aleasrv.cs.unitn.it/techalea.nsf/FA28AACD2A0B4147C12568FC0058AD11/$FILE/Sguera2.pdf

## Monte Carlo 

# <span style="color:blue">Method 1: Monte Carlo simulation</span>

### <span style="color:blue">Explain that the value of the option should be $$V_t = e^{−r(T−t)}\mathbb{E}[(S_T − K)^+\mathbf{1}_{S_u \ge B,\forall u \in (t,T)}|\mathcal{F}_t]$$ where $\mathbb{E}$ is in the risk neutral measure.</span>

### <span style="color:blue">Using the SDE, write a program that computes one trajectory of $S_t$ for $t < T$; it should return a list of values $S_0, S_{\Delta t}, S_{2 \Delta t}, ..., S_T$ for $\Delta T = \frac{T}{N}$ where $N$ is the number of points (you can choose N = 252 for example).</span>

In [73]:
def SGenerate(t,T,N,r,sigma,x):
    import numpy as np
    step = (T-t)/N
    S = x
    S_path = [S]
    for i in range(N):
        dw = np.random.normal(0, np.sqrt(step))
        dS = r*S*step+sigma*S*dw
        S += dS
        S_path.append(S) 
    return S_path

In [74]:
SGenerate(0,1,252,0.02,0.2,100)

[100,
 99.97007104914394,
 99.8184289332495,
 101.9824621224973,
 102.73414335813705,
 101.96545545550053,
 102.52153958262855,
 103.31631712121974,
 103.82250950440577,
 102.69047551525293,
 101.65298494210857,
 101.88869739681766,
 100.3489617555841,
 98.74134898385921,
 98.34946534248812,
 97.40557850285751,
 96.59252002180942,
 97.52994390802233,
 99.98565594554701,
 100.19409957378274,
 102.69278468504332,
 102.56770150937511,
 101.27618663854136,
 100.92859587183023,
 99.50195120968452,
 99.59894877772562,
 100.78866449136298,
 98.7828651180742,
 98.40857591700991,
 96.44868392532031,
 96.3160902751553,
 97.49700226258928,
 99.94623617141633,
 98.13312412548379,
 98.12641251365288,
 99.1410272169116,
 98.84059529133452,
 97.39084586289444,
 96.05612234567148,
 97.10334752642365,
 98.61245325511669,
 97.92402074252357,
 97.86255402716162,
 97.20653872287976,
 96.42124566852871,
 99.12186399020233,
 99.52990697933681,
 99.985129588958,
 101.08887572331882,
 101.82985344584839,
 101

In [75]:
def Payoff(t,T,N,r,sigma,x,B,K):
    S_path = SGenerate(t,T,N,r,sigma,x)
    if min(S_path) > B:
        return max(S_path[-1]-K,0),S_path
    else:
        return 0,S_path
    
# example
Payoff(0,1,252,0.02,0.2,100,80,110)[0]

0

In [76]:
def MC(Np,t,T,N,r,sigma,x,B,K):
    import numpy as np
    payoffs = []
    price = 0
    for j in range(Np):
        payoff = Payoff(t,T,N,r,sigma,x,B,K)[0]
        price += payoff
        payoffs.append(payoff)
    price = price*np.exp(-r*(T-t))/Np
    return price,payoffs

In [77]:
MC(1000,0,1,360000,0.01,0.2,100,90,100)

(np.float64(7.249901002953328),
 [47.69205248581909,
  0,
  0,
  0,
  0,
  28.013952680548584,
  0,
  0,
  23.589091546492767,
  0,
  0,
  0,
  34.11050453724616,
  0,
  0,
  34.54039764869174,
  0,
  0,
  0,
  0,
  0,
  21.169564051723654,
  21.255976622200194,
  0,
  0,
  11.267230828415876,
  0,
  0,
  0,
  0,
  0,
  26.835617822705856,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  12.096228112680222,
  0,
  22.657918961577423,
  9.343827634232937,
  11.84291409348269,
  0,
  0,
  27.891192480634032,
  0,
  62.01063150274058,
  0,
  30.2796707425276,
  0,
  50.03585521626218,
  0,
  18.579624385017993,
  0,
  0,
  10.861040127267103,
  0,
  0,
  0,
  0,
  0,
  0,
  33.72675156872151,
  0,
  0,
  25.296602174471985,
  0,
  1.4497490658132932,
  16.449831069655872,
  0,
  0,
  1.0068755754748082,
  0,
  0,
  0,
  0,
  25.412215085794912,
  0,
  14.7067962436669,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  20.06584146973674,
  0,
  0,
  29.18033802996112,
  57.96662869580021,


for monte carlo provide mean and variance

monte carlo is higher! non sa cosa fa il browniano tra un giorno e l'altro (fa interpolazione)
monte carlo è sbagliato per questo tipo di opzioni!!!! 